In [1]:
import pandas as pd
import altair as alt

# Load the data from the URL
url = 'https://github.com/UIUC-iSchool-DataViz/is445_data/raw/main/licenses_fall2022.csv'
df = pd.read_csv(url)

# Display basic information about the dataset
print(df.dtypes)
display(df.head())

_id                                          int64
License Type                                object
Description                                 object
License Number                              object
License Status                              object
Business                                    object
Title                                       object
First Name                                  object
Middle                                      object
Last Name                                   object
Prefix                                      object
Suffix                                      object
Business Name                               object
BusinessDBA                                 object
Original Issue Date                         object
Effective Date                              object
Expiration Date                             object
City                                        object
State                                       object
Zip                            

,_id,License Type,Description,License Number,License Status,Business,Title,First Name,Middle,Last Name,...,Specialty/Qualifier,Controlled Substance Schedule,Delegated Controlled Substance Schedule,Ever Disciplined,LastModifiedDate,Case Number,Action,Discipline Start Date,Discipline End Date,Discipline Reason
0,1189509,DETECTIVE BOARD,PERMANENT EMPLOYEE REGISTRATION,129446286,NOT RENEWED,N,NaN,EILEEN,NaN,SANTACRUZ,...,NaN,NaN,NaN,N,03/18/2022,NaN,NaN,NaN,NaN,NaN
1,801037,DETECTIVE BOARD,FIREARM CONTROL CARD,229030294.0,NOT RENEWED,N,NaN,DAGMAR,J,NORDLUND,...,NaN,NaN,NaN,N,08/16/2006,NaN,NaN,NaN,NaN,NaN
2,365129,COSMO,LICENSED COSMETOLOGIST,11053076.0,NOT RENEWED,N,NaN,RADOJE,NaN,ZELENOVIC,...,NaN,NaN,NaN,N,05/26/2006,NaN,NaN,NaN,NaN,NaN
3,595427,COSMO,LICENSED COSMETOLOGIST,11295645.0,ACTIVE,N,NaN,BECKY SUE,L,BURROUGHS,...,NaN,NaN,NaN,N,11/12/2021,NaN,NaN,NaN,NaN,NaN
4,653668,COSMO,LICENSED NAIL TECHNICIAN,169006247,NOT RENEWED,N,NaN,BILL G,L,LETNER,...,NaN,NaN,NaN,N,05/30/2006,NaN,NaN,NaN,NaN,NaN


In [ ]:
display(df.describe())

In [2]:
print(df.isnull().sum())

_id                                            0
License Type                                   0
Description                                    0
License Number                                60
License Status                                 0
Business                                       0
Title                                       9890
First Name                                   395
Middle                                      6378
Last Name                                    395
Prefix                                      9997
Suffix                                      9590
Business Name                                  0
BusinessDBA                                 9885
Original Issue Date                            5
Effective Date                               792
Expiration Date                              500
City                                          11
State                                          0
Zip                                           71
County              

In [3]:
# group for type and status
license_counts = df.groupby(['License Type', 'License Status']).size().reset_index(name='Count')

top_license_types = df['License Type'].value_counts().head(15).index
license_counts_filtered = license_counts[license_counts['License Type'].isin(top_license_types)]

print(license_counts)

             License Type License Status  Count
0               APPRAISAL        EXPIRED      5
1               ARCHITECT         ACTIVE      8
2               ARCHITECT       DECEASED      1
3               ARCHITECT       INACTIVE      5
4               ARCHITECT    NOT RENEWED      6
..                    ...            ...    ...
100  MASSAGE LICENSING BD       INACTIVE     12
101  MASSAGE LICENSING BD    NOT RENEWED     17
102         MEDICAL BOARD         ACTIVE      1
103         MEDICAL BOARD       INACTIVE      2
104         MEDICAL BOARD    NOT RENEWED      3

[105 rows x 3 columns]


In [4]:
# Horizontal bar
chart1 = alt.Chart(license_counts_filtered).mark_bar().encode(
    y=alt.Y('License Type:N', sort='-x', title='License Type'),
    x=alt.X('Count:Q', title='Number of Licenses'),
    color=alt.Color('License Status:N', title='License Status', scale=alt.Scale(scheme='category10')),
    tooltip=[
        alt.Tooltip('License Type:N', title='License Type'),
        alt.Tooltip('License Status:N', title='Status'),
        alt.Tooltip('Count:Q', title='Count', format=',')
    ]
).properties(
    title='Distribution of License Types by Status',
    width=600,
    height=400
).interactive()

chart1

alt.Chart(...)

In [13]:
#only get the top 10 to simplify
top_10_licenses = df['License Type'].value_counts().head(10).index
discipline_simple = df[df['License Type'].isin(top_10_licenses)].groupby(['License Type', 'Ever Disciplined']).size().reset_index(name='Count')
print(discipline_simple.head())

       License Type Ever Disciplined  Count
0         APPRAISAL                N      5
1         ARCHITECT                N     20
2  ATHLETIC TRAINER                N     13
3  ATHLETIC TRAINER                Y      1
4         ATHLETICS                N     15


In [15]:
# Create a simple grouped bar chart with hover highlighting
highlight = alt.selection_point(on='mouseover', fields=['Ever Disciplined'], nearest=True)

chart2 = alt.Chart(discipline_simple).mark_bar().encode(
    x=alt.X('License Type:N', 
            title='License Type',
            axis=alt.Axis(labelAngle=-45),
            sort='-y'),
    y=alt.Y('Count:Q', title='Number of Licenses'),
    color=alt.Color('Ever Disciplined:N', 
                    title='Ever Disciplined?',
                    scale=alt.Scale(domain=['Yes', 'No'], range=['#e74c3c', '#2ecc71'])),
    xOffset='Ever Disciplined:N',
    opacity=alt.condition(highlight, alt.value(1), alt.value(0.5)),
    tooltip=[
        alt.Tooltip('License Type:N', title='License Type'),
        alt.Tooltip('Ever Disciplined:N', title='Ever Disciplined?'),
        alt.Tooltip('Count:Q', title='Count', format=',')
    ]
).add_params(
    highlight
).properties(
    title='Discipline Status Across Top 10 License Types',
    width=600,
    height=400
)

chart2

alt.Chart(...)